# Model Training: Linear Regression

## Project Overview
This notebook trains a Linear Regression model to predict car prices based on the engineered features from the previous notebook. We'll cover train-test splitting, feature scaling, model training, and evaluation.

## Table of Contents
1. Import Libraries
2. Load Processed Data
3. Train-Test Split
4. Feature Scaling (Standardization)
5. Train Linear Regression Model
6. Model Evaluation
7. Interpretation of Results
8. Residual Analysis
9. Save Model (Optional)

## 1. Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Scikit-learn modules
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## 2. Load Processed Data

Load the feature-engineered dataset from the previous notebook.

In [ ]:
# Load the feature-engineered dataset
auto = pd.read_csv("data/processed/auto_features.csv")

print(f"Dataset shape: {auto.shape}")
print(f"\nFirst 5 rows:")
auto.head()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(auto.isnull().sum())

# Check data types
print("\nData types:")
print(auto.dtypes.value_counts())

## 3. Train-Test Split

**Why split the data?**
- **Training set**: Used to train the model (learn patterns)
- **Testing set**: Used to evaluate model performance on unseen data
- **Prevents overfitting**: Ensures the model generalizes well

**Parameters:**
- `test_size=0.2`: 20% of data for testing, 80% for training
- `random_state=0`: Ensures reproducible results

In [ ]:
# Separate features (X) and target (y)
X = auto.drop("price", axis=1)
y = auto["price"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures ({len(X.columns)} total):")
print(X.columns.tolist())

In [ ]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print(f"Training set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set size: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Verify no data leakage
print(f"\nTraining target range: ${y_train.min():,.0f} - ${y_train.max():,.0f}")
print(f"Testing target range: ${y_test.min():,.0f} - ${y_test.max():,.0f}")

## 4. Feature Scaling (Standardization)

**Why scale features?**
- Linear Regression assumes features are on similar scales
- StandardScaler transforms data to have **mean = 0** and **standard deviation = 1**
- Formula: `z = (x - μ) / σ`
- Prevents features with larger magnitudes from dominating the model

**Important**: Fit on training data ONLY, then transform both train and test

In [ ]:
# Initialize the scaler
scaler = StandardScaler()

# Fit on training data and transform
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier inspection (optional)
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Scaling complete!")
print(f"\nTraining data - mean: {X_train_scaled.mean():.10f}, std: {X_train_scaled.std():.3f}")
print(f"Testing data - mean: {X_test_scaled.mean():.10f}, std: {X_test_scaled.std():.3f}")

# Show before/after scaling for first few rows
print("\n" + "="*60)
print("Before scaling (first 5 rows, first 5 features):")
print("="*60)
print(X_train.iloc[:5, :5])

print("\n" + "="*60)
print("After scaling (first 5 rows, first 5 features):")
print("="*60)
print(X_train_scaled_df.iloc[:5, :5])

## 5. Train Linear Regression Model

**Linear Regression Formula:**
$$ y = \beta_0 + \beta_1x_1 + \beta_2x_2 + ... + \beta_nx_n $$

Where:
- `y` = predicted price
- `β₀` = intercept
- `β₁...βₙ` = coefficients for each feature
- `x₁...xₙ` = feature values

**How it works:** Finds the line/plane that minimizes the sum of squared residuals (Ordinary Least Squares)

In [ ]:
# Create and train the model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Model training complete!")
print(f"Intercept (β₀): {model.intercept_:.2f}")
print(f"Number of features: {len(model.coef_)}")

## 6. Model Evaluation

In [ ]:
# Calculate R-squared (Coefficient of Determination)
train_r2 = model.score(X_train_scaled, y_train)
test_r2 = model.score(X_test_scaled, y_test)

print('=' * 60)
print('MODEL PERFORMANCE METRICS')
print('=' * 60)
print(f'Coefficient of Determination (R²) - Train: {train_r2:.3f}')
print(f'Coefficient of Determination (R²) - Test: {test_r2:.3f}')
print(f'Difference (Train - Test): {(train_r2 - test_r2):.3f}')

# Additional metrics
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print(f'\nMean Squared Error (MSE) - Train: {train_mse:.2f}')
print(f'Mean Squared Error (MSE) - Test: {test_mse:.2f}')
print(f'\nRoot Mean Squared Error (RMSE) - Train: ${train_rmse:.2f}')
print(f'Root Mean Squared Error (RMSE) - Test: ${test_rmse:.2f}')
print(f'\nMean Absolute Error (MAE) - Train: ${train_mae:.2f}')
print(f'Mean Absolute Error (MAE) - Test: ${test_mae:.2f}')

**Interpretation of Metrics:**

| Metric | Range | Interpretation |
|--------|-------|----------------|
| **R²** | 0 to 1 | Proportion of variance explained. Higher is better |
| **RMSE** | 0 to ∞ | Average prediction error in original units (dollars) |
| **MAE** | 0 to ∞ | Average absolute error (less sensitive to outliers) |

## 7. Regression Coefficients & Feature Importance

**What do coefficients mean?**
- For scaled features: A 1-standard-deviation increase in the feature leads to a coefficient change in price
- **Positive coefficient**: Feature increases with price
- **Negative coefficient**: Feature decreases with price

In [ ]:
# Create Series with coefficients and feature names
coefficients = pd.Series(model.coef_, index=X.columns)

print('=' * 60)
print('REGRESSION COEFFICIENTS (Standardized)')
print('=' * 60)
print(f'Intercept: {model.intercept_:.2f}\n')
print(coefficients.sort_values(ascending=False))

In [ ]:
# Visualize top positive and negative coefficients
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Top 10 positive coefficients
top_coef = coefficients.nlargest(10)
axes[0].barh(range(len(top_coef)), top_coef.values)
axes[0].set_yticks(range(len(top_coef)))
axes[0].set_yticklabels(top_coef.index)
axes[0].set_xlabel('Coefficient Value')
axes[0].set_title('Top 10 Features Increasing Price')
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# Top 10 negative coefficients (most negative)
bottom_coef = coefficients.nsmallest(10)
axes[1].barh(range(len(bottom_coef)), bottom_coef.values)
axes[1].set_yticks(range(len(bottom_coef)))
axes[1].set_yticklabels(bottom_coef.index)
axes[1].set_xlabel('Coefficient Value')
axes[1].set_title('Top 10 Features Decreasing Price')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 8. Residual Analysis

**What are residuals?**
- Residual = Actual Price - Predicted Price
- Good model: Residuals should be randomly distributed around zero
- Patterns in residuals suggest model misspecification

In [ ]:
# Calculate residuals
train_residuals = y_train - y_train_pred
test_residuals = y_test - y_test_pred

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Residuals vs Predicted (Train)
axes[0,0].scatter(y_train_pred, train_residuals, alpha=0.5)
axes[0,0].axhline(y=0, color='red', linestyle='--')
axes[0,0].set_xlabel('Predicted Price')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Training: Residuals vs Predicted')

# Residuals vs Predicted (Test)
axes[0,1].scatter(y_test_pred, test_residuals, alpha=0.5)
axes[0,1].axhline(y=0, color='red', linestyle='--')
axes[0,1].set_xlabel('Predicted Price')
axes[0,1].set_ylabel('Residuals')
axes[0,1].set_title('Testing: Residuals vs Predicted')

# Histogram of residuals (Train)
axes[1,0].hist(train_residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1,0].axvline(x=0, color='red', linestyle='--')
axes[1,0].set_xlabel('Residuals')
axes[1,0].set_ylabel('Frequency')
axes[1,0].set_title('Training: Residual Distribution')

# Histogram of residuals (Test)
axes[1,1].hist(test_residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1,1].axvline(x=0, color='red', linestyle='--')
axes[1,1].set_xlabel('Residuals')
axes[1,1].set_ylabel('Frequency')
axes[1,1].set_title('Testing: Residual Distribution')

plt.tight_layout()
plt.show()

# Residual statistics
print("Residual Statistics:")
print(f"Training - Mean: {train_residuals.mean():.2f}, Std: {train_residuals.std():.2f}")
print(f"Testing  - Mean: {test_residuals.mean():.2f}, Std: {test_residuals.std():.2f}")

## 9. Actual vs Predicted Plot

Visualizes how well predictions match actual values. Points should ideally fall along the diagonal line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.5)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title(f'Training Set (R² = {train_r2:.3f})')

# Testing set
axes[1].scatter(y_test, y_test_pred, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].set_title(f'Testing Set (R² = {test_r2:.3f})')

plt.tight_layout()
plt.show()

## 10. Model Performance Summary

**Key Findings:**

| Metric | Training | Testing | Interpretation |
|--------|----------|---------|----------------|
| R² | {train_r2:.3f} | {test_r2:.3f} | {'✓ Good fit' if train_r2 > 0.7 else '⚠️ Low explanatory power'} |
| RMSE | ${train_rmse:.0f} | ${test_rmse:.0f} | Average prediction error |
| MAE | ${train_mae:.0f} | ${test_mae:.0f} | Typical absolute error |

**Overfitting Check:**
- Train R² - Test R² = {(train_r2 - test_r2):.3f}
- {'⚠️ Potential overfitting' if (train_r2 - test_r2) > 0.1 else '✓ No significant overfitting detected'}

In [ ]:
print('=' * 60)
print('MODEL SUMMARY')
print('=' * 60)
print(f'Model Type: Linear Regression')
print(f'Training samples: {len(y_train)}')
print(f'Testing samples: {len(y_test)}')
print(f'Features used: {len(X.columns)}')
print(f'\nR² Score: {test_r2:.3f}')
print(f'RMSE: ${test_rmse:.2f}')
print(f'MAE: ${test_mae:.2f}')

# Top 3 most influential features
print(f'\nTop 3 features increasing price:')
for feature, coef in coefficients.nlargest(3).items():
    print(f'  • {feature}: +${coef:.0f} per std dev')

print(f'\nTop 3 features decreasing price:')
for feature, coef in coefficients.nsmallest(3).items():
    print(f'  • {feature}: ${coef:.0f} per std dev')

print('=' * 60)

## 11. Save Model (Optional)

Save the trained model and scaler for future use.

In [ ]:
import joblib

# Save model and scaler
joblib.dump(model, 'models/linear_regression_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print("Model saved to 'models/linear_regression_model.pkl'")
print("Scaler saved to 'models/scaler.pkl'")

# Optional: Save predictions
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_test_pred,
    'Residual': test_residuals
})
results.to_csv('predictions/test_predictions.csv', index=False)
print("Test predictions saved to 'predictions/test_predictions.csv'")

In [ ]:
# Check if directories exist, create if not
import os
os.makedirs('models', exist_ok=True)
os.makedirs('predictions', exist_ok=True)
print("Directories created: 'models/', 'predictions/'")

## Next Steps & Improvements

**Potential improvements to try:**

1. **Regularization** (Ridge, Lasso) to reduce overfitting
2. **Polynomial features** to capture non-linear relationships
3. **Cross-validation** for more robust evaluation
4. **Other algorithms**: Random Forest, Gradient Boosting, XGBoost
5. **Hyperparameter tuning** with GridSearchCV
6. **Feature selection** using RFE or feature importance

**Next Notebook Ideas:**
- Model Comparison (Linear vs Ridge vs Lasso vs Random Forest)
- Hyperparameter Tuning with Cross-Validation
- Feature Importance Analysis with Tree-based Models

In [ ]:
print("Training notebook complete!")